# 🎯 Notebook 01: Exploración de Datos IDEAM## Proyecto de Tesis - Conectividad Hidráulica Río Magdalena**Objetivo:** Aprender a cargar, limpiar y explorar datos de nivel de agua usando Pandas.**Instrucciones:** Corre cada celda con `Shift + Enter` o el botón ▶️.

## 0. Verificar que todo funcionaPrimero comprobamos que Pandas esté instalado y listo.

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltprint("✅ Todo listo para la tesis!")print(f"Pandas version: {pd.__version__}")print(f"NumPy version: {np.__version__}")

## 1. Crear datos de ejemplo (simulando IDEAM)Como aún no tienes datos reales, generamos una serie sintética.Cuando descargues del IDEAM, reemplazarás esta sección por `pd.read_csv()`.

In [ ]:
np.random.seed(42)fechas = pd.date_range(start="2014-01-01", end="2024-12-31", freq="D")n = len(fechas)# Simular nivel con ciclo estacional + El Niñodia_año = fechas.dayofyear.valuesciclo = 2.5 * np.sin(2 * np.pi * (dia_año - 120) / 365)nivel_base = 85.0en_2015_2016 = ((fechas >= "2015-06-01") & (fechas <= "2016-05-31")).astype(float)en_2023_2024 = ((fechas >= "2023-05-01") & (fechas <= "2024-04-30")).astype(float)efecto_en = -1.8 * en_2015_2016 - 1.2 * en_2023_2024nivel = nivel_base + ciclo + efecto_en + np.random.normal(0, 0.25, n)mascara_nan = np.random.rand(n) < 0.05nivel[mascara_nan] = np.nancalidad = np.where(np.isnan(nivel), 9, np.random.choice([0,1,2], n, p=[0.7,0.2,0.1]))df = pd.DataFrame({    "Fecha": fechas,    "Nivel_msnm": np.round(nivel, 3),    "Calidad": calidad})print(f"📊 {len(df)} registros generados")print(f"❓ {df['Nivel_msnm'].isna().sum()} valores faltantes")df.head()

## 2. Inspeccionar los datosRevisamos tipos, estadísticas y rangos antes de tocar nada.

In [ ]:
print("--- Tipos de datos ---")print(df.dtypes)print("\n--- Estadísticas ---")print(df["Nivel_msnm"].describe())print(f"\n--- Rango de fechas ---")print(f"Desde: {df['Fecha'].min().date()}")print(f"Hasta: {df['Fecha'].max().date()}")

## 3. Limpiar los datosEliminamos faltantes y creamos columnas útiles.

In [ ]:
df_limpio = df.dropna(subset=["Nivel_msnm"]).copy()df_limpio = df_limpio[df_limpio["Calidad"] <= 2].copy()df_limpio = df_limpio.sort_values("Fecha").reset_index(drop=True)df_limpio["Año"] = df_limpio["Fecha"].dt.yeardf_limpio["Mes"] = df_limpio["Fecha"].dt.monthprint(f"Registros limpios: {len(df_limpio)}")df_limpio.head()

## 4. Análisis de ConectividadAplicamos la regla de tu tesis:- Si Nivel > Z_fondo → CONECTADO 🟢- Si Nivel <= Z_fondo → DESCONECTADO 🔴

In [ ]:
Z_FONDO = 83.5  # Reemplazar con tu cota de campodf_limpio["Estado"] = np.where(df_limpio["Nivel_msnm"] > Z_FONDO, "CONECTADO", "DESCONECTADO")# Resumen por añoresumen = df_limpio.groupby("Año").agg(    dias_totales=("Estado", "size"),    dias_desconectados=("Estado", lambda x: (x == "DESCONECTADO").sum()),    nivel_min=("Nivel_msnm", "min"),    nivel_prom=("Nivel_msnm", "mean")).reset_index()resumen["pct_desconectado"] = (resumen["dias_desconectados"] / resumen["dias_totales"] * 100).round(1)print(resumen.to_string(index=False))

## 5. VisualizaciónGraficamos la serie con el umbral de desconexión.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))ax.plot(df_limpio["Fecha"], df_limpio["Nivel_msnm"], color="steelblue", linewidth=0.8, label="Nivel IDEAM")ax.axhline(y=Z_FONDO, color="red", linestyle="--", linewidth=2, label=f"Z_fondo = {Z_FONDO} msnm")ax.fill_between(df_limpio["Fecha"], df_limpio["Nivel_msnm"].min()-0.5, Z_FONDO,                where=(df_limpio["Nivel_msnm"] <= Z_FONDO), color="red", alpha=0.2, label="Zona desconexion")ax.set_xlabel("Fecha")ax.set_ylabel("Nivel (msnm)")ax.set_title("Serie Temporal - Analisis de Conectividad Hidraulica")ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## ✅ Siguientes pasos1. Ajusta `Z_FONDO` y observa cómo cambia el % de desconexión.2. Descarga datos reales del IDEAM (DHIME) y reemplaza la sección 1.3. Guarda este notebook: `Ctrl + S`.